In [1]:
import torch
import torch.nn as nn

torch.manual_seed(42)

# ── Data ───

In [2]:
# Fake dataset: y = 3x₁ + 2x₂ + noise  (regression problem)
X = torch.rand(200, 2)                        # 200 samples, 2 features
y = 3 * X[:, 0:1] + 2 * X[:, 1:2] + 0.1 * torch.randn(200, 1)

In [3]:
# Train/validation split — 160 train, 40 val
X_train, X_val = X[:160],  X[160:]
y_train, y_val = y[:160],  y[160:]

# ── Model, loss, optimizer ──

In [4]:
model     = nn.Linear(2, 1)                   # simple linear model
loss_fn   = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

# ── Training loop ──

In [5]:
EPOCHS     = 50
BATCH_SIZE = 32

for epoch in range(EPOCHS):

    # ── Training phase ──
    model.train()   # tells PyTorch "we are training" (more on this shortly)

    # Split data into batches manually for now
    for i in range(0, len(X_train), BATCH_SIZE):
        X_batch = X_train[i : i + BATCH_SIZE]  # grab one batch
        y_batch = y_train[i : i + BATCH_SIZE]

        # The 4-step heartbeat
        predictions = model(X_batch)            # ① forward pass
        loss        = loss_fn(predictions, y_batch)  # ② compute loss
        optimizer.zero_grad()                   # ③ zero gradients
        loss.backward()                         # ④ backward pass
        optimizer.step()                        # ⑤ update weights

    # ── Validation phase ──
    model.eval()    # tells PyTorch "we are evaluating"

    with torch.no_grad():   # disable gradient tracking — saves memory
        val_preds = model(X_val)
        val_loss  = loss_fn(val_preds, y_val)

    # Print progress every 10 epochs
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1:3d}/{EPOCHS} | "
              f"Train loss: {loss.item():.4f} | "
              f"Val loss: {val_loss.item():.4f}")

Epoch  10/50 | Train loss: 4.8786 | Val loss: 4.4519
Epoch  20/50 | Train loss: 2.1427 | Val loss: 1.9738
Epoch  30/50 | Train loss: 0.9110 | Val loss: 0.8776
Epoch  40/50 | Train loss: 0.4488 | Val loss: 0.4773
Epoch  50/50 | Train loss: 0.2999 | Val loss: 0.3521


In [6]:
with torch.no_grad():
    val_preds = model(X_val)   # no gradient tracking — evaluation only

In [7]:
val_preds.shape

torch.Size([40, 1])

In [8]:
train_losses = []
val_losses   = []

for epoch in range(EPOCHS):

    # --- training phase (same as before) ---
    model.train()
    for i in range(0, len(X_train), BATCH_SIZE):
        X_batch = X_train[i : i + BATCH_SIZE]
        y_batch = y_train[i : i + BATCH_SIZE]
        predictions = model(X_batch)
        loss        = loss_fn(predictions, y_batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    # --- validation phase ---
    model.eval()
    with torch.no_grad():
        train_loss = loss_fn(model(X_train), y_train)  # full training loss
        val_loss   = loss_fn(model(X_val),   y_val)

    train_losses.append(train_loss.item())
    val_losses.append(val_loss.item())
